# Notebook 16: Knowledge Graph Analysis — Cross-Disease Gene Network & Drug Repurposing

**Framework:** [pathway-subtyping](https://pypi.org/project/pathway-subtyping/) v0.3.0

## Overview

This notebook constructs a biological knowledge graph (KG) from gene contributions across **6 GEO datasets** (1,075 samples) to identify:

1. **Hub genes** — high-centrality genes bridging ASD and SCZ molecular subtypes
2. **Network communities** — PPI-based gene modules with cross-disease enrichment
3. **Drug repurposing candidates** — data-driven therapeutic targets ranked by network centrality
4. **Mechanistic convergence** — topology-aware evidence for shared ASD↔SCZ biology

### Data Sources
| Source | Type | Details |
|--------|------|---------|
| Gene contributions (6 datasets) | CSV | Effect sizes from subtype characterization |
| ASD pathways | GMT | 15 pathways, curated from SFARI/ASC |
| SCZ pathways | GMT | 14 pathways, curated from PGC3/SCHEMA |
| STRING v12.0 | API | Human protein-protein interactions |
| DGIdb | API | Drug-gene interaction database |

### Outline
- **Phase A (Cells 1-8):** Data loading, gene statistics, tier selection
- **Phase B (Cells 9-12):** STRING PPI network retrieval and analysis
- **Phase C (Cells 13-18):** KG construction, centrality, community detection
- **Phase D (Cells 19-25):** Drug repurposing, network figures, export

### Dependencies
- Notebooks 10-15 (gene contribution CSVs in `research-results/`)
- `networkx>=3.3`, `matplotlib`, `seaborn`, `pandas`, `numpy`, `requests`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import time
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import requests
import seaborn as sns
import networkx as nx
from networkx.algorithms.community import louvain_communities

print(f"networkx {nx.__version__}")
print(f"pandas {pd.__version__}")
print(f"numpy {np.__version__}")

# Paths (relative to examples/notebooks/)
RESULTS_DIR = Path('../../research-results')
PATHWAYS_DIR = Path('../../data/pathways')
OUTPUT_DIR = Path('outputs/knowledge_graph')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Verify paths
assert RESULTS_DIR.exists(), f"Missing: {RESULTS_DIR}"
assert PATHWAYS_DIR.exists(), f"Missing: {PATHWAYS_DIR}"

print(f"Results directory: {RESULTS_DIR.resolve()}")
print(f"Pathways directory: {PATHWAYS_DIR.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

In [ ]:
# === Load gene contributions from 6 primary datasets ===

GENE_CONTRIB_FILES = {
    "GSE28521": RESULTS_DIR / "GSE28521/gene_contributions.csv",
    "GSE64018": RESULTS_DIR / "GSE64018/frontal-cortex/gene_contributions.csv",
    "GSE80655": RESULTS_DIR / "GSE80655/gene_contributions.csv",
    "GSE53987": RESULTS_DIR / "GSE53987/gene_contributions.csv",
    "GSE111175": RESULTS_DIR / "GSE111175/gene_contributions.csv",
    "GSE18123": RESULTS_DIR / "GSE18123/gene_contributions.csv",
}

DISEASE_MAP = {
    "GSE28521": "ASD", "GSE64018": "ASD", "GSE80655": "SCZ",
    "GSE53987": "SCZ", "GSE111175": "ASD", "GSE18123": "ASD",
}

TISSUE_MAP = {
    "GSE28521": "Brain", "GSE64018": "Brain", "GSE80655": "Brain",
    "GSE53987": "Brain", "GSE111175": "Blood", "GSE18123": "Blood",
}

dfs = []
for dataset, path in GENE_CONTRIB_FILES.items():
    if not path.exists():
        print(f"WARNING: Missing {path}")
        continue
    df = pd.read_csv(path)
    df['dataset'] = dataset
    df['disease'] = DISEASE_MAP[dataset]
    df['tissue'] = TISSUE_MAP[dataset]
    dfs.append(df)
    print(f"  {dataset}: {len(df):,} rows, {df['gene'].nunique()} unique genes ({DISEASE_MAP[dataset]}, {TISSUE_MAP[dataset]})")

all_contributions = pd.concat(dfs, ignore_index=True)
print(f"\nTotal: {len(all_contributions):,} gene-subtype-pathway rows across {len(dfs)} datasets")
print(f"Unique genes in contributions: {all_contributions['gene'].nunique()}")

In [ ]:
# === Compute per-gene statistics across datasets ===

gene_dataset_stats = all_contributions.groupby('gene').agg(
    mean_abs_effect_size=('effect_size', lambda x: np.abs(x).mean()),
    max_abs_effect_size=('effect_size', lambda x: np.abs(x).max()),
    n_entries=('effect_size', 'count'),
    n_datasets=('dataset', 'nunique'),
    datasets=('dataset', lambda x: ','.join(sorted(x.unique()))),
    diseases=('disease', lambda x: ','.join(sorted(x.unique()))),
).reset_index()

# Disease association
def get_disease_assoc(diseases_str):
    diseases = set(diseases_str.split(','))
    if diseases == {'ASD', 'SCZ'}:
        return 'Both'
    elif 'ASD' in diseases:
        return 'ASD'
    else:
        return 'SCZ'

gene_dataset_stats['disease_association'] = gene_dataset_stats['diseases'].apply(get_disease_assoc)

# Top pathway per gene (by mean |effect_size|)
top_pw = all_contributions.copy()
top_pw['abs_es'] = top_pw['effect_size'].abs()
top_pw = top_pw.groupby(['gene', 'pathway'])['abs_es'].mean().reset_index()
top_pw = top_pw.sort_values('abs_es', ascending=False).drop_duplicates('gene')
gene_dataset_stats = gene_dataset_stats.merge(
    top_pw[['gene', 'pathway']].rename(columns={'pathway': 'top_pathway'}),
    on='gene', how='left'
)

gene_dataset_stats = gene_dataset_stats.sort_values('mean_abs_effect_size', ascending=False).reset_index(drop=True)

# Summary
assoc_counts = gene_dataset_stats['disease_association'].value_counts()
print("Disease association of genes in contributions:")
for assoc, count in assoc_counts.items():
    print(f"  {assoc}: {count}")

print(f"\nTop 20 genes by mean |effect_size|:")
print(gene_dataset_stats[['gene', 'mean_abs_effect_size', 'n_datasets', 'disease_association', 'top_pathway']].head(20).to_string(index=False))

In [ ]:
# === Load ASD and SCZ pathway GMT files ===

def load_gmt(path):
    """Parse GMT file: pathway_name<TAB>url<TAB>gene1<TAB>gene2<TAB>..."""
    pathways = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split('\t')
            name = parts[0]
            genes = [g for g in parts[2:] if g]  # skip URL column
            pathways[name] = genes
    return pathways

asd_pathways = load_gmt(PATHWAYS_DIR / 'autism_pathways.gmt')
scz_pathways = load_gmt(PATHWAYS_DIR / 'schizophrenia_pathways.gmt')

print(f"ASD pathways: {len(asd_pathways)}")
for pw, genes in asd_pathways.items():
    print(f"  {pw}: {len(genes)} genes")

print(f"\nSCZ pathways: {len(scz_pathways)}")
for pw, genes in scz_pathways.items():
    print(f"  {pw}: {len(genes)} genes")

# Shared pathways
shared_pathways = set(asd_pathways.keys()) & set(scz_pathways.keys())
asd_only_pathways = set(asd_pathways.keys()) - shared_pathways
scz_only_pathways = set(scz_pathways.keys()) - shared_pathways

print(f"\nShared pathways ({len(shared_pathways)}): {sorted(shared_pathways)}")
print(f"ASD-only pathways ({len(asd_only_pathways)}): {sorted(asd_only_pathways)}")
print(f"SCZ-only pathways ({len(scz_only_pathways)}): {sorted(scz_only_pathways)}")
print(f"Total unique pathways: {len(asd_pathways) + len(scz_pathways) - len(shared_pathways)}")

In [ ]:
# === Build gene universe from GMT files ===

# Collect all genes and their pathway memberships
gene_to_pathways = defaultdict(set)
gene_to_disease_source = defaultdict(set)  # which GMT file(s) the gene comes from

for pw, genes in asd_pathways.items():
    for g in genes:
        gene_to_pathways[g].add(pw)
        gene_to_disease_source[g].add('ASD')

for pw, genes in scz_pathways.items():
    for g in genes:
        gene_to_pathways[g].add(pw)
        gene_to_disease_source[g].add('SCZ')

all_pathway_genes = set(gene_to_pathways.keys())

# Disease association from GMT membership
gene_gmt_disease = {}
for g in all_pathway_genes:
    sources = gene_to_disease_source[g]
    if sources == {'ASD', 'SCZ'}:
        gene_gmt_disease[g] = 'Both'
    elif 'ASD' in sources:
        gene_gmt_disease[g] = 'ASD'
    else:
        gene_gmt_disease[g] = 'SCZ'

# Venn-style counts
gmt_assoc_counts = Counter(gene_gmt_disease.values())
asd_genes = {g for g, d in gene_gmt_disease.items() if d in ('ASD', 'Both')}
scz_genes = {g for g, d in gene_gmt_disease.items() if d in ('SCZ', 'Both')}
both_genes = {g for g, d in gene_gmt_disease.items() if d == 'Both'}

print(f"Total unique genes in GMT files: {len(all_pathway_genes)}")
print(f"  ASD-only genes: {gmt_assoc_counts.get('ASD', 0)}")
print(f"  SCZ-only genes: {gmt_assoc_counts.get('SCZ', 0)}")
print(f"  Both (shared): {gmt_assoc_counts.get('Both', 0)}")
print(f"\nGenes in ASD pathways: {len(asd_genes)}")
print(f"Genes in SCZ pathways: {len(scz_genes)}")
print(f"Genes in both: {len(both_genes)}")

# Merge GMT disease info with contribution stats
gene_stats = pd.DataFrame({
    'gene': list(all_pathway_genes),
    'gmt_disease': [gene_gmt_disease[g] for g in all_pathway_genes],
    'n_pathways': [len(gene_to_pathways[g]) for g in all_pathway_genes],
})

# Left-join contribution stats (some GMT genes may not appear in contributions)
gene_stats = gene_stats.merge(
    gene_dataset_stats[['gene', 'mean_abs_effect_size', 'max_abs_effect_size',
                         'n_datasets', 'disease_association', 'top_pathway']],
    on='gene', how='left'
)
gene_stats['mean_abs_effect_size'] = gene_stats['mean_abs_effect_size'].fillna(0)
gene_stats['n_datasets'] = gene_stats['n_datasets'].fillna(0).astype(int)
gene_stats = gene_stats.sort_values('mean_abs_effect_size', ascending=False).reset_index(drop=True)

# Genes with contribution data vs GMT-only
n_with_data = (gene_stats['n_datasets'] > 0).sum()
print(f"\nGenes with contribution data: {n_with_data} / {len(gene_stats)}")
print(f"Genes in GMT only (no contribution data): {len(gene_stats) - n_with_data}")

## Gene Selection for Core Knowledge Graph

All genes from the ASD + SCZ pathway GMT files form the KG gene universe. For visualization focus, we define two tiers:

- **Tier 1 (Core):** Top 50 genes ranked by mean |effect_size| across datasets — labeled in figures, prioritized for drug lookup
- **Tier 2 (Extended):** Remaining pathway genes — included in KG as smaller nodes

This ensures the KG captures the full pathway biology while highlighting the genes with strongest subtype-discriminating signal.

In [ ]:
# === Assign gene tiers ===

# Tier 1: top 50 by mean |effect_size| (must have contribution data)
gene_stats['tier'] = 'Tier 2'
tier1_mask = gene_stats['n_datasets'] > 0
tier1_genes = gene_stats[tier1_mask].nlargest(50, 'mean_abs_effect_size').index
gene_stats.loc[tier1_genes, 'tier'] = 'Tier 1'

tier1 = gene_stats[gene_stats['tier'] == 'Tier 1']
tier2 = gene_stats[gene_stats['tier'] == 'Tier 2']

print(f"Tier 1 (core): {len(tier1)} genes")
print(f"Tier 2 (extended): {len(tier2)} genes")
print(f"Total: {len(gene_stats)} genes\n")

# Tier 1 disease breakdown
t1_assoc = tier1['gmt_disease'].value_counts()
print("Tier 1 disease breakdown:")
for assoc, count in t1_assoc.items():
    print(f"  {assoc}: {count}")

print(f"\nTier 1 genes (top 50 by mean |effect_size|):")
display_cols = ['gene', 'mean_abs_effect_size', 'n_datasets', 'gmt_disease', 'n_pathways', 'top_pathway']
print(tier1[display_cols].to_string(index=False))

## STRING Protein-Protein Interaction Network

We query the [STRING database](https://string-db.org/) (v12.0) API to retrieve protein-protein interactions among our pathway genes.

- **Species:** 9606 (Homo sapiens)
- **Score threshold:** 400 (medium confidence)
- **Method:** POST to STRING API with gene identifiers (avoids downloading 300MB+ full network)
- **Output:** Gene-gene PPI edges with combined confidence scores

Cross-disease PPI edges (connecting ASD-pathway genes to SCZ-pathway genes) are particularly informative — they reveal physical protein interactions bridging the two conditions.

In [ ]:
# === Query STRING API for PPI among our pathway genes ===

CACHE_FILE = OUTPUT_DIR / 'string_ppi_edges.csv'
gene_list = list(all_pathway_genes)

if CACHE_FILE.exists():
    print(f"Loading cached STRING data from {CACHE_FILE}")
    string_edges = pd.read_csv(CACHE_FILE)
    print(f"Loaded {len(string_edges)} PPI edges")
else:
    print(f"Querying STRING API for {len(gene_list)} genes...")
    # STRING API accepts up to ~2000 identifiers per request
    # Batch if needed (we have ~350 genes, well under limit)
    url = "https://string-db.org/api/tsv/network"

    # Use POST for large gene lists
    params = {
        "identifiers": "\r".join(gene_list),
        "species": 9606,
        "required_score": 400,
        "caller_identity": "pathway_subtyping_framework",
    }

    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = requests.post(url, data=params, timeout=120)
            response.raise_for_status()
            break
        except (requests.exceptions.RequestException, requests.exceptions.Timeout) as e:
            print(f"  Attempt {attempt + 1}/{max_retries} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(5 * (attempt + 1))
            else:
                raise RuntimeError(f"STRING API failed after {max_retries} attempts") from e

    # Parse TSV response
    from io import StringIO
    raw = pd.read_csv(StringIO(response.text), sep='\t')
    print(f"Raw STRING response: {len(raw)} rows, columns: {list(raw.columns)}")

    # Extract gene symbols and scores
    # STRING returns preferredName_A, preferredName_B, score
    string_edges = raw[['preferredName_A', 'preferredName_B', 'score']].copy()
    string_edges.columns = ['gene1', 'gene2', 'combined_score']

    # Remove self-loops and duplicates
    string_edges = string_edges[string_edges['gene1'] != string_edges['gene2']]
    # Ensure undirected: keep one direction per pair
    string_edges['pair'] = string_edges.apply(
        lambda r: tuple(sorted([r['gene1'], r['gene2']])), axis=1
    )
    string_edges = string_edges.drop_duplicates('pair').drop(columns='pair').reset_index(drop=True)

    string_edges.to_csv(CACHE_FILE, index=False)
    print(f"\nSaved {len(string_edges)} unique PPI edges to {CACHE_FILE}")

# Filter to genes in our pathway set
mask = string_edges['gene1'].isin(all_pathway_genes) & string_edges['gene2'].isin(all_pathway_genes)
string_edges = string_edges[mask].reset_index(drop=True)
print(f"PPI edges between pathway genes: {len(string_edges)}")

# Genes with at least one PPI edge
ppi_genes = set(string_edges['gene1']) | set(string_edges['gene2'])
print(f"Genes with PPI data: {len(ppi_genes)} / {len(all_pathway_genes)} ({100*len(ppi_genes)/len(all_pathway_genes):.1f}%)")
print(f"Genes without PPI: {len(all_pathway_genes - ppi_genes)}")

In [ ]:
# === PPI summary statistics ===

# Degree distribution
degree_counts = Counter()
for _, row in string_edges.iterrows():
    degree_counts[row['gene1']] += 1
    degree_counts[row['gene2']] += 1

degrees = pd.Series(degree_counts)
print(f"PPI degree statistics:")
print(f"  Mean degree: {degrees.mean():.1f}")
print(f"  Median degree: {degrees.median():.0f}")
print(f"  Max degree: {degrees.max()} ({degrees.idxmax()})")
print(f"  Min degree: {degrees.min()}")

# Cross-disease PPI edges
cross_disease_edges = []
for _, row in string_edges.iterrows():
    d1 = gene_gmt_disease.get(row['gene1'], 'Unknown')
    d2 = gene_gmt_disease.get(row['gene2'], 'Unknown')
    if d1 != d2 and 'Unknown' not in (d1, d2):
        # At least one is ASD-only and the other is SCZ-only
        if (d1 in ('ASD',) and d2 in ('SCZ',)) or (d1 in ('SCZ',) and d2 in ('ASD',)):
            cross_disease_edges.append(row)

print(f"\nCross-disease PPI edges (ASD-only ↔ SCZ-only): {len(cross_disease_edges)}")

# Score distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(string_edges['combined_score'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_xlabel('STRING Combined Score')
axes[0].set_ylabel('Edge Count')
axes[0].set_title('PPI Score Distribution')
axes[0].axvline(x=700, color='red', linestyle='--', alpha=0.7, label='High confidence (700)')
axes[0].legend()

axes[1].hist(degrees, bins=30, color='coral', edgecolor='white')
axes[1].set_xlabel('Degree')
axes[1].set_ylabel('Gene Count')
axes[1].set_title('PPI Degree Distribution')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'ppi_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/ppi_distributions.png")

In [ ]:
# === Pathway crosstalk matrix (PPI connections between pathway gene sets) ===

# Build gene-to-pathway lookup (a gene can belong to multiple pathways)
all_pw = {}
for pw, genes in asd_pathways.items():
    all_pw[pw] = set(genes)
for pw, genes in scz_pathways.items():
    if pw in all_pw:
        all_pw[pw] = all_pw[pw] | set(genes)  # merge shared pathway gene lists
    else:
        all_pw[pw] = set(genes)

pw_names = sorted(all_pw.keys())
n_pw = len(pw_names)

# Count PPI edges between each pathway pair
crosstalk = np.zeros((n_pw, n_pw), dtype=int)
for _, row in string_edges.iterrows():
    g1, g2 = row['gene1'], row['gene2']
    for i, pw1 in enumerate(pw_names):
        if g1 not in all_pw[pw1]:
            continue
        for j, pw2 in enumerate(pw_names):
            if g2 not in all_pw[pw2]:
                continue
            if i <= j:  # count each edge once per pathway pair
                crosstalk[i, j] += 1
                if i != j:
                    crosstalk[j, i] += 1

pathway_crosstalk = pd.DataFrame(crosstalk, index=pw_names, columns=pw_names)
pathway_crosstalk.to_csv(OUTPUT_DIR / 'pathway_crosstalk_matrix.csv')

# Top pathway pairs by PPI connections
pairs = []
for i in range(n_pw):
    for j in range(i + 1, n_pw):
        if crosstalk[i, j] > 0:
            pairs.append((pw_names[i], pw_names[j], crosstalk[i, j]))
pairs.sort(key=lambda x: -x[2])

print(f"Top 15 pathway pairs by PPI connections:")
for pw1, pw2, count in pairs[:15]:
    shared_tag = " *" if pw1 in shared_pathways and pw2 in shared_pathways else ""
    print(f"  {pw1} ↔ {pw2}: {count} edges{shared_tag}")
print("\n* = both pathways shared between ASD and SCZ")
print(f"\nSaved: {OUTPUT_DIR}/pathway_crosstalk_matrix.csv")

## Knowledge Graph Construction

We build a NetworkX `MultiDiGraph` with typed nodes and edges:

**Node types:**
- `Gene` — pathway genes with effect size, disease association, tier
- `Pathway` — ASD/SCZ/shared pathways with gene counts
- `Disease` — ASD, SCZ

**Edge types:**
- `GENE_IN_PATHWAY` — gene membership in pathway
- `GENE_INTERACTS` — protein-protein interaction (STRING)
- `GENE_ASSOCIATED_DISEASE` — gene → disease link (weighted by effect size)
- `PATHWAY_ASSOCIATED_DISEASE` — pathway → disease link

Network centrality (betweenness, PageRank) identifies **hub genes** — high-centrality nodes that bridge multiple pathways and/or diseases. Community detection reveals **network modules** that may correspond to distinct molecular mechanisms.

In [ ]:
# === Build Knowledge Graph ===

G = nx.MultiDiGraph()

# Add Disease nodes
for disease in ['ASD', 'SCZ']:
    G.add_node(disease, node_type='Disease')

# Add Pathway nodes
for pw_name in pw_names:
    pw_disease = 'Both' if pw_name in shared_pathways else ('ASD' if pw_name in asd_pathways else 'SCZ')
    G.add_node(pw_name, node_type='Pathway', disease=pw_disease, n_genes=len(all_pw[pw_name]))

# Add Gene nodes
for _, row in gene_stats.iterrows():
    g = row['gene']
    G.add_node(g, node_type='Gene',
               gmt_disease=row['gmt_disease'],
               mean_effect_size=row['mean_abs_effect_size'],
               n_datasets=int(row['n_datasets']),
               n_pathways=int(row['n_pathways']),
               tier=row['tier'])

# Add GENE_IN_PATHWAY edges
n_gip = 0
for pw_name, pw_genes in all_pw.items():
    for g in pw_genes:
        if g in G:
            G.add_edge(g, pw_name, edge_type='GENE_IN_PATHWAY', weight=1.0)
            n_gip += 1

# Add GENE_INTERACTS edges (PPI)
n_ppi = 0
for _, row in string_edges.iterrows():
    g1, g2 = row['gene1'], row['gene2']
    if g1 in G and g2 in G:
        w = row['combined_score'] / 1000.0
        G.add_edge(g1, g2, edge_type='GENE_INTERACTS', weight=w)
        G.add_edge(g2, g1, edge_type='GENE_INTERACTS', weight=w)
        n_ppi += 1

# Add GENE_ASSOCIATED_DISEASE edges
n_gad = 0
for _, row in gene_stats.iterrows():
    g = row['gene']
    if row['mean_abs_effect_size'] > 0:
        if row['gmt_disease'] in ('ASD', 'Both'):
            G.add_edge(g, 'ASD', edge_type='GENE_ASSOCIATED_DISEASE', weight=row['mean_abs_effect_size'])
            n_gad += 1
        if row['gmt_disease'] in ('SCZ', 'Both'):
            G.add_edge(g, 'SCZ', edge_type='GENE_ASSOCIATED_DISEASE', weight=row['mean_abs_effect_size'])
            n_gad += 1

# Add PATHWAY_ASSOCIATED_DISEASE edges
n_pad = 0
for pw_name in pw_names:
    if pw_name in asd_pathways:
        G.add_edge(pw_name, 'ASD', edge_type='PATHWAY_ASSOCIATED_DISEASE', weight=1.0)
        n_pad += 1
    if pw_name in scz_pathways:
        G.add_edge(pw_name, 'SCZ', edge_type='PATHWAY_ASSOCIATED_DISEASE', weight=1.0)
        n_pad += 1

# Summary
node_types = Counter(data.get('node_type', 'Unknown') for _, data in G.nodes(data=True))
print("Knowledge Graph Summary:")
print(f"  Total nodes: {G.number_of_nodes()}")
for nt, count in sorted(node_types.items()):
    print(f"    {nt}: {count}")
print(f"  Total edges: {G.number_of_edges()}")
print(f"    GENE_IN_PATHWAY: {n_gip}")
print(f"    GENE_INTERACTS: {n_ppi} (×2 directed)")
print(f"    GENE_ASSOCIATED_DISEASE: {n_gad}")
print(f"    PATHWAY_ASSOCIATED_DISEASE: {n_pad}")

In [ ]:
# === Centrality metrics on gene PPI subgraph ===

# Extract gene-only undirected PPI subgraph for centrality analysis
gene_nodes = [n for n, d in G.nodes(data=True) if d.get('node_type') == 'Gene']
ppi_edges = [(u, v) for u, v, d in G.edges(data=True) if d.get('edge_type') == 'GENE_INTERACTS']

# Build undirected graph for centrality
G_ppi = nx.Graph()
G_ppi.add_nodes_from(gene_nodes)
for u, v in ppi_edges:
    if not G_ppi.has_edge(u, v):
        # Get weight from original graph
        edge_data = G.get_edge_data(u, v)
        w = 1.0
        if edge_data:
            for key, d in edge_data.items():
                if d.get('edge_type') == 'GENE_INTERACTS':
                    w = d.get('weight', 1.0)
                    break
        G_ppi.add_edge(u, v, weight=w)

print(f"Gene PPI subgraph: {G_ppi.number_of_nodes()} nodes, {G_ppi.number_of_edges()} edges")

# Compute centrality metrics
print("Computing centrality metrics...")
degree_cent = nx.degree_centrality(G_ppi)
betweenness_cent = nx.betweenness_centrality(G_ppi, weight='weight', seed=42)
pagerank = nx.pagerank(G_ppi, weight='weight')

# Clustering coefficient
clustering = nx.clustering(G_ppi, weight='weight')

# Build centrality DataFrame
gene_centrality = pd.DataFrame({
    'gene': gene_nodes,
    'degree': [G_ppi.degree(n) for n in gene_nodes],
    'degree_centrality': [degree_cent.get(n, 0) for n in gene_nodes],
    'betweenness': [betweenness_cent.get(n, 0) for n in gene_nodes],
    'pagerank': [pagerank.get(n, 0) for n in gene_nodes],
    'clustering_coeff': [clustering.get(n, 0) for n in gene_nodes],
})

# Merge with gene stats
gene_centrality = gene_centrality.merge(
    gene_stats[['gene', 'gmt_disease', 'mean_abs_effect_size', 'n_datasets', 'tier', 'n_pathways']],
    on='gene', how='left'
)
gene_centrality = gene_centrality.sort_values('betweenness', ascending=False).reset_index(drop=True)

print(f"\nTop 15 genes by betweenness centrality:")
print(gene_centrality[['gene', 'degree', 'betweenness', 'pagerank', 'gmt_disease', 'mean_abs_effect_size', 'tier']].head(15).to_string(index=False))

In [ ]:
# === Hub gene identification ===

# Hub genes: top 20 by betweenness centrality
hub_genes = gene_centrality.nlargest(20, 'betweenness').copy()

# Cross-disease bridging genes: high betweenness + disease_association = "Both"
hub_genes['is_bridge'] = hub_genes['gmt_disease'] == 'Both'

# Annotate with pathway memberships
hub_genes['pathways'] = hub_genes['gene'].apply(
    lambda g: ', '.join(sorted(gene_to_pathways.get(g, set())))
)

print("=" * 80)
print("TOP 20 HUB GENES (ranked by betweenness centrality)")
print("=" * 80)
for i, (_, row) in enumerate(hub_genes.iterrows()):
    bridge_tag = " ** BRIDGE **" if row['is_bridge'] else ""
    print(f"\n{i+1}. {row['gene']}{bridge_tag}")
    print(f"   Disease: {row['gmt_disease']} | Degree: {row['degree']} | "
          f"Betweenness: {row['betweenness']:.4f} | PageRank: {row['pagerank']:.4f}")
    print(f"   Effect size: {row['mean_abs_effect_size']:.3f} | Datasets: {row['n_datasets']}")
    print(f"   Pathways: {row['pathways']}")

n_bridges = hub_genes['is_bridge'].sum()
print(f"\n{'=' * 80}")
print(f"Cross-disease bridging genes in top 20: {n_bridges}")
print(f"{'=' * 80}")

# Save hub genes
hub_genes.to_csv(OUTPUT_DIR / 'hub_genes_ranked.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR}/hub_genes_ranked.csv")

In [ ]:
# === Community detection on gene PPI subgraph ===

# Louvain community detection (built-in to networkx >= 3.3)
print("Running Louvain community detection (resolution=1.0)...")
communities = louvain_communities(G_ppi, resolution=1.0, seed=42)
print(f"Found {len(communities)} communities\n")

# Assign community labels
gene_community = {}
for i, comm in enumerate(communities):
    for gene in comm:
        gene_community[gene] = i

# Analyze each community
community_data = []
for i, comm in enumerate(communities):
    comm_genes = list(comm)
    n_genes = len(comm_genes)
    if n_genes < 3:
        continue

    # Disease composition
    diseases = [gene_gmt_disease.get(g, 'Unknown') for g in comm_genes]
    disease_counts = Counter(diseases)
    pct_asd = 100 * (disease_counts.get('ASD', 0) + disease_counts.get('Both', 0)) / n_genes
    pct_scz = 100 * (disease_counts.get('SCZ', 0) + disease_counts.get('Both', 0)) / n_genes
    pct_both = 100 * disease_counts.get('Both', 0) / n_genes

    # Top pathways in this community
    pw_counter = Counter()
    for g in comm_genes:
        for pw in gene_to_pathways.get(g, []):
            pw_counter[pw] += 1
    top_pws = [pw for pw, _ in pw_counter.most_common(3)]

    # Is this a cross-disease community? (has both ASD-only and SCZ-only genes)
    has_asd_only = disease_counts.get('ASD', 0) > 0
    has_scz_only = disease_counts.get('SCZ', 0) > 0
    is_cross = has_asd_only and has_scz_only

    # Hub genes in this community
    comm_hubs = [g for g in comm_genes if g in hub_genes['gene'].values]

    community_data.append({
        'community_id': i,
        'n_genes': n_genes,
        'pct_asd': round(pct_asd, 1),
        'pct_scz': round(pct_scz, 1),
        'pct_both': round(pct_both, 1),
        'is_cross_disease': is_cross,
        'top_pathways': '; '.join(top_pws),
        'hub_genes': '; '.join(comm_hubs) if comm_hubs else 'None',
        'asd_only': disease_counts.get('ASD', 0),
        'scz_only': disease_counts.get('SCZ', 0),
        'both': disease_counts.get('Both', 0),
    })

community_df = pd.DataFrame(community_data).sort_values('n_genes', ascending=False).reset_index(drop=True)
n_cross = community_df['is_cross_disease'].sum()

print(f"Communities with ≥3 genes: {len(community_df)}")
print(f"Cross-disease communities: {n_cross}\n")

print(community_df[['community_id', 'n_genes', 'pct_asd', 'pct_scz', 'pct_both',
                      'is_cross_disease', 'top_pathways', 'hub_genes']].to_string(index=False))

community_df.to_csv(OUTPUT_DIR / 'community_analysis.csv', index=False)
print(f"\nSaved: {OUTPUT_DIR}/community_analysis.csv")

In [ ]:
# === Network statistics and convergence subnetwork ===

# Connected components
components = list(nx.connected_components(G_ppi))
components.sort(key=len, reverse=True)
print(f"Connected components: {len(components)}")
print(f"Largest component: {len(components[0])} genes")
if len(components) > 1:
    print(f"Isolated nodes: {sum(1 for c in components if len(c) == 1)}")

# Statistics on largest component
lcc = G_ppi.subgraph(components[0])
print(f"\nLargest Connected Component (LCC):")
print(f"  Nodes: {lcc.number_of_nodes()}")
print(f"  Edges: {lcc.number_of_edges()}")
print(f"  Density: {nx.density(lcc):.4f}")
print(f"  Transitivity: {nx.transitivity(lcc):.4f}")
print(f"  Avg clustering: {nx.average_clustering(lcc):.4f}")

# Diameter and avg shortest path (may be slow for large graphs)
if lcc.number_of_nodes() <= 500:
    diameter = nx.diameter(lcc)
    avg_path = nx.average_shortest_path_length(lcc)
    print(f"  Diameter: {diameter}")
    print(f"  Avg shortest path: {avg_path:.2f}")

# Disease assortativity (do same-disease genes preferentially interact?)
disease_attr = {n: gene_gmt_disease.get(n, 'Unknown') for n in G_ppi.nodes()}
nx.set_node_attributes(G_ppi, disease_attr, 'disease')
try:
    assort = nx.attribute_assortativity_coefficient(G_ppi, 'disease')
    print(f"  Disease assortativity: {assort:.4f}")
    if assort > 0:
        print("    (Positive = same-disease genes preferentially interact)")
    else:
        print("    (Negative = cross-disease interactions more common than expected)")
except:
    print("  Disease assortativity: could not compute")

# === Convergence subnetwork ===
# Nodes: genes with gmt_disease="Both" OR genes bridging ASD-only and SCZ-only communities
bridge_genes = set()

# 1. Genes in both disease pathways
for g in all_pathway_genes:
    if gene_gmt_disease.get(g) == 'Both':
        bridge_genes.add(g)

# 2. Genes with PPI edges to both ASD-only and SCZ-only genes
for g in G_ppi.nodes():
    neighbors = set(G_ppi.neighbors(g))
    neighbor_diseases = {gene_gmt_disease.get(n) for n in neighbors}
    if 'ASD' in neighbor_diseases and 'SCZ' in neighbor_diseases:
        bridge_genes.add(g)

# Build convergence subgraph (bridge genes + their PPI edges)
bridge_genes_in_ppi = bridge_genes & set(G_ppi.nodes())
convergence_subgraph = G_ppi.subgraph(bridge_genes_in_ppi).copy()

print(f"\nConvergence Subnetwork:")
print(f"  Bridge genes: {len(bridge_genes_in_ppi)}")
print(f"  PPI edges: {convergence_subgraph.number_of_edges()}")

# Export to GraphML
nx.write_graphml(convergence_subgraph, str(OUTPUT_DIR / 'convergence_subnetwork.graphml'))
print(f"  Saved: {OUTPUT_DIR}/convergence_subnetwork.graphml")

# Also export full gene PPI graph
for n in G_ppi.nodes():
    G_ppi.nodes[n]['gmt_disease'] = gene_gmt_disease.get(n, 'Unknown')
    G_ppi.nodes[n]['tier'] = gene_stats.set_index('gene').loc[n, 'tier'] if n in gene_stats['gene'].values else 'Tier 2'
    G_ppi.nodes[n]['community'] = gene_community.get(n, -1)
nx.write_graphml(G_ppi, str(OUTPUT_DIR / 'cross_disease_network.graphml'))
print(f"  Saved: {OUTPUT_DIR}/cross_disease_network.graphml")

## Drug Repurposing via DGIdb

The [Drug-Gene Interaction Database (DGIdb)](https://dgidb.org/) aggregates drug-gene interactions from >30 sources (DrugBank, PharmGKB, ChEMBL, etc.).

We query DGIdb for drug interactions targeting our **hub genes** (top 20 by betweenness centrality) plus additional **Tier 1 core genes**. Drugs are ranked by the network centrality of their target genes — drugs targeting high-centrality hub genes rank highest because they impact the most connected nodes in the cross-disease network.

**Repurposing rationale categories:**
- **Cross-disease hub target:** Drug targets a gene with `disease_association=Both` and high betweenness
- **Disease-specific hub target:** Drug targets a top-10 gene in its disease subnetwork
- **Convergent pathway target:** Drug targets a gene in ≥2 shared ASD↔SCZ pathways

In [ ]:
# === Query DGIdb GraphQL API for drug-gene interactions ===

DRUG_CACHE = OUTPUT_DIR / 'dgidb_interactions.csv'

# Query top hub genes + tier 1 genes
query_genes = list(set(
    hub_genes['gene'].tolist() +
    gene_centrality[gene_centrality['tier'] == 'Tier 1']['gene'].tolist()
))
print(f"Querying DGIdb for {len(query_genes)} genes...")

if DRUG_CACHE.exists():
    print(f"Loading cached DGIdb data from {DRUG_CACHE}")
    drug_interactions = pd.read_csv(DRUG_CACHE)
else:
    # DGIdb GraphQL API (v2 REST API is deprecated)
    url = "https://dgidb.org/api/graphql"

    all_interactions = []
    # Batch in groups of 25 genes per GraphQL query
    batch_size = 25
    for i in range(0, len(query_genes), batch_size):
        batch = query_genes[i:i+batch_size]
        gene_list_str = ', '.join(f'"{g}"' for g in batch)

        query = f'''{{
          genes(names: [{gene_list_str}]) {{
            nodes {{
              name
              interactions {{
                drug {{
                  name
                  approved
                }}
                interactionScore
                interactionTypes {{
                  type
                  directionality
                }}
                sources {{
                  fullName
                }}
              }}
            }}
          }}
        }}'''

        for attempt in range(3):
            try:
                response = requests.post(url, json={'query': query}, timeout=60)
                response.raise_for_status()
                data = response.json()
                if 'errors' in data:
                    raise ValueError(f"GraphQL errors: {data['errors']}")
                break
            except Exception as e:
                print(f"  Batch {i//batch_size + 1} attempt {attempt+1} failed: {e}")
                if attempt < 2:
                    time.sleep(3 * (attempt + 1))
                else:
                    print(f"  Skipping batch after 3 failures")
                    data = {'data': {'genes': {'nodes': []}}}

        genes_data = data.get('data', {}).get('genes', {}).get('nodes', [])
        for gene_entry in genes_data:
            gene_name = gene_entry.get('name', '')
            for interaction in gene_entry.get('interactions', []):
                drug_info = interaction.get('drug', {})
                int_types = interaction.get('interactionTypes', [])
                sources = interaction.get('sources', [])
                all_interactions.append({
                    'gene': gene_name,
                    'drug': drug_info.get('name', ''),
                    'approved': drug_info.get('approved', False),
                    'interaction_type': int_types[0].get('type', 'unknown') if int_types else 'unknown',
                    'directionality': int_types[0].get('directionality', '') if int_types else '',
                    'n_sources': len(sources),
                    'sources': '; '.join(s.get('fullName', '') for s in sources),
                    'score': interaction.get('interactionScore', 0),
                })

        print(f"  Batch {i//batch_size + 1}: {len(genes_data)} genes returned")
        if i + batch_size < len(query_genes):
            time.sleep(0.5)  # rate limiting

    drug_interactions = pd.DataFrame(all_interactions)
    if len(drug_interactions) > 0:
        drug_interactions.to_csv(DRUG_CACHE, index=False)
        print(f"\nSaved {len(drug_interactions)} interactions to {DRUG_CACHE}")

if len(drug_interactions) > 0:
    print(f"\nDGIdb results:")
    print(f"  Total interactions: {len(drug_interactions)}")
    print(f"  Unique drugs: {drug_interactions['drug'].nunique()}")
    print(f"  Unique genes with drug targets: {drug_interactions['gene'].nunique()}")
    print(f"  Approved drugs: {drug_interactions['approved'].sum() if 'approved' in drug_interactions.columns else 'N/A'}")
    print(f"  Interaction types: {dict(drug_interactions['interaction_type'].value_counts().head(10))}")
else:
    print("\nNo drug interactions found (DGIdb may be unavailable)")

In [ ]:
# === Build drug repurposing table ===

if len(drug_interactions) > 0:
    # Merge with centrality data
    drug_table = drug_interactions.merge(
        gene_centrality[['gene', 'betweenness', 'pagerank', 'degree', 'gmt_disease',
                          'mean_abs_effect_size', 'n_pathways']],
        on='gene', how='left'
    )

    # Assign repurposing rationale
    betweenness_median = gene_centrality['betweenness'].median()

    def get_rationale(row):
        reasons = []
        if row['gmt_disease'] == 'Both' and row['betweenness'] > betweenness_median:
            reasons.append('Cross-disease hub target')
        if row['betweenness'] > gene_centrality['betweenness'].quantile(0.9):
            reasons.append('Top 10% centrality')
        if row['n_pathways'] >= 2:
            reasons.append(f"Multi-pathway ({int(row['n_pathways'])} pathways)")
        if not reasons:
            reasons.append('Pathway gene target')
        return '; '.join(reasons)

    drug_table['rationale'] = drug_table.apply(get_rationale, axis=1)

    # Rank by target gene betweenness
    drug_table = drug_table.sort_values('betweenness', ascending=False).reset_index(drop=True)

    # Deduplicate: keep highest-scoring interaction per drug-gene pair
    drug_table = drug_table.sort_values(['betweenness', 'n_sources'], ascending=[False, False])
    drug_table_dedup = drug_table.drop_duplicates(['drug', 'gene']).reset_index(drop=True)

    # Top 30 drug candidates
    top_drugs = drug_table_dedup.head(30)

    print("=" * 100)
    print("TOP 30 DRUG REPURPOSING CANDIDATES (ranked by target gene centrality)")
    print("=" * 100)
    for i, (_, row) in enumerate(top_drugs.iterrows()):
        print(f"\n{i+1}. {row['drug']}")
        print(f"   Target: {row['gene']} ({row['gmt_disease']}) | "
              f"Mechanism: {row['interaction_type']} | Sources: {row['n_sources']}")
        print(f"   Betweenness: {row['betweenness']:.4f} | PageRank: {row['pagerank']:.4f}")
        print(f"   Rationale: {row['rationale']}")

    # Save full table
    drug_table_dedup.to_csv(OUTPUT_DIR / 'drug_repurposing_table.csv', index=False)
    print(f"\nSaved: {OUTPUT_DIR}/drug_repurposing_table.csv")
    print(f"Total unique drug-gene pairs: {len(drug_table_dedup)}")
else:
    print("Skipping drug table (no DGIdb data available)")
    drug_table_dedup = pd.DataFrame()

In [ ]:
# === Manuscript-ready network figure ===

fig, ax = plt.subplots(1, 1, figsize=(16, 16))

# Layout: spring layout with seed
pos = nx.spring_layout(G_ppi, k=0.5, iterations=80, seed=42, weight='weight')

# Node colors by disease association
color_map = {'ASD': '#4A90D9', 'SCZ': '#D94A4A', 'Both': '#9B59B6', 'Unknown': '#CCCCCC'}
node_colors = [color_map.get(gene_gmt_disease.get(n, 'Unknown'), '#CCCCCC') for n in G_ppi.nodes()]

# Node sizes: proportional to betweenness centrality
min_size, max_size = 15, 400
bw_values = np.array([betweenness_cent.get(n, 0) for n in G_ppi.nodes()])
if bw_values.max() > 0:
    node_sizes = min_size + (max_size - min_size) * (bw_values / bw_values.max())
else:
    node_sizes = np.full(len(bw_values), min_size)

# Edge widths: proportional to STRING score
edge_weights = [G_ppi[u][v].get('weight', 0.4) for u, v in G_ppi.edges()]
edge_widths = [0.2 + 1.5 * w for w in edge_weights]

# Draw edges (light gray)
nx.draw_networkx_edges(G_ppi, pos, ax=ax, edge_color='#CCCCCC', width=edge_widths, alpha=0.3)

# Draw nodes
nx.draw_networkx_nodes(G_ppi, pos, ax=ax, node_color=node_colors, node_size=node_sizes,
                        edgecolors='white', linewidths=0.5, alpha=0.85)

# Labels: only Tier 1 genes
tier1_genes_set = set(gene_stats[gene_stats['tier'] == 'Tier 1']['gene'])
hub_gene_set = set(hub_genes['gene'])
labels = {}
for n in G_ppi.nodes():
    if n in hub_gene_set:
        labels[n] = n
    elif n in tier1_genes_set and betweenness_cent.get(n, 0) > np.percentile(bw_values[bw_values > 0], 50):
        labels[n] = n

# Draw hub labels bold, others normal
hub_labels = {n: l for n, l in labels.items() if n in hub_gene_set}
other_labels = {n: l for n, l in labels.items() if n not in hub_gene_set}

nx.draw_networkx_labels(G_ppi, pos, hub_labels, ax=ax, font_size=8, font_weight='bold', font_color='black')
nx.draw_networkx_labels(G_ppi, pos, other_labels, ax=ax, font_size=6, font_color='#333333')

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#4A90D9', label='ASD pathway genes'),
    mpatches.Patch(facecolor='#D94A4A', label='SCZ pathway genes'),
    mpatches.Patch(facecolor='#9B59B6', label='Shared (ASD + SCZ)'),
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11, framealpha=0.9)

ax.set_title('Cross-Disease Gene Network: ASD ↔ SCZ Pathway Interactions',
             fontsize=14, fontweight='bold', pad=20)
ax.axis('off')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'network_figure.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(OUTPUT_DIR / 'network_figure.pdf', bbox_inches='tight', facecolor='white')
plt.show()
print(f"Saved: {OUTPUT_DIR}/network_figure.png (300 DPI)")
print(f"Saved: {OUTPUT_DIR}/network_figure.pdf (vector)")

In [ ]:
# === Drug targets overlay figure ===

if len(drug_table_dedup) > 0:
    fig, ax = plt.subplots(1, 1, figsize=(16, 16))

    # Use same layout as main figure
    # Draw base gene network (lighter)
    nx.draw_networkx_edges(G_ppi, pos, ax=ax, edge_color='#E0E0E0', width=0.3, alpha=0.2)
    nx.draw_networkx_nodes(G_ppi, pos, ax=ax, node_color=node_colors, node_size=node_sizes * 0.5,
                            edgecolors='white', linewidths=0.3, alpha=0.4)

    # Top drug per unique target gene (one drug per gene for visual spread)
    top_drug_entries = drug_table_dedup.drop_duplicates('gene').head(10)
    drug_colors = plt.cm.Greens(np.linspace(0.4, 0.9, len(top_drug_entries)))

    # Add drug nodes and edges
    drug_pos = {}
    for idx, (_, drug_row) in enumerate(top_drug_entries.iterrows()):
        drug_name = drug_row['drug']
        target_gene = drug_row['gene']

        if target_gene not in pos:
            continue

        # Position drug node near its target
        angle = 2 * np.pi * idx / len(top_drug_entries)
        offset = 0.15
        drug_pos[drug_name] = (
            pos[target_gene][0] + offset * np.cos(angle),
            pos[target_gene][1] + offset * np.sin(angle),
        )

        # Draw drug node (diamond)
        ax.scatter(*drug_pos[drug_name], s=200, c=[drug_colors[idx]],
                   marker='D', edgecolors='darkgreen', linewidths=1.5, zorder=5)
        ax.annotate(drug_name, drug_pos[drug_name], fontsize=6,
                    fontweight='bold', color='darkgreen',
                    ha='center', va='bottom', xytext=(0, 8),
                    textcoords='offset points')

        # Draw drug→gene edge
        ax.annotate('', xy=pos[target_gene], xytext=drug_pos[drug_name],
                    arrowprops=dict(arrowstyle='->', color='green', lw=1.5, alpha=0.7))

        # Highlight target gene
        ax.scatter(*pos[target_gene], s=node_sizes[list(G_ppi.nodes()).index(target_gene)] * 1.5,
                   facecolors='none', edgecolors='green', linewidths=2, zorder=4)
        ax.annotate(target_gene, pos[target_gene], fontsize=7, fontweight='bold',
                    ha='center', va='top', xytext=(0, -10), textcoords='offset points')

    # Legend
    legend_elements = [
        mpatches.Patch(facecolor='#4A90D9', label='ASD genes'),
        mpatches.Patch(facecolor='#D94A4A', label='SCZ genes'),
        mpatches.Patch(facecolor='#9B59B6', label='Shared genes'),
        plt.Line2D([0], [0], marker='D', color='w', markerfacecolor='green',
                   markersize=10, label='Drug candidates'),
    ]
    ax.legend(handles=legend_elements, loc='upper left', fontsize=11, framealpha=0.9)

    ax.set_title('Drug Repurposing Targets in Cross-Disease Network',
                 fontsize=14, fontweight='bold', pad=20)
    ax.axis('off')

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'drug_network_figure.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f"Saved: {OUTPUT_DIR}/drug_network_figure.png")
else:
    print("Skipping drug overlay figure (no DGIdb data)")

In [ ]:
# === Pathway crosstalk heatmap ===

fig, ax = plt.subplots(1, 1, figsize=(14, 12))

# Annotate pathway names with disease tag
pw_labels = []
for pw in pw_names:
    if pw in shared_pathways:
        pw_labels.append(f"{pw} [Both]")
    elif pw in asd_pathways:
        pw_labels.append(f"{pw} [ASD]")
    else:
        pw_labels.append(f"{pw} [SCZ]")

# Mask diagonal for clarity
mask = np.eye(n_pw, dtype=bool)
crosstalk_display = pathway_crosstalk.copy()
crosstalk_display.index = pw_labels
crosstalk_display.columns = pw_labels

sns.heatmap(crosstalk_display, mask=mask, cmap='YlOrRd', annot=True, fmt='d',
            xticklabels=True, yticklabels=True, ax=ax, linewidths=0.5,
            cbar_kws={'label': 'PPI Edge Count'})

ax.set_title('Pathway Crosstalk via Protein-Protein Interactions', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=8)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pathway_crosstalk_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"Saved: {OUTPUT_DIR}/pathway_crosstalk_heatmap.png")

In [ ]:
# === Results summary and export ===

# Build summary
kg_summary = {
    "notebook": "16_knowledge_graph_analysis",
    "framework_version": "0.3.0",
    "n_datasets": len(GENE_CONTRIB_FILES),
    "datasets": list(GENE_CONTRIB_FILES.keys()),
    "n_genes_gmt": len(all_pathway_genes),
    "n_pathways_unique": len(pw_names),
    "n_pathways_asd": len(asd_pathways),
    "n_pathways_scz": len(scz_pathways),
    "n_pathways_shared": len(shared_pathways),
    "n_ppi_edges": len(string_edges),
    "ppi_coverage_pct": round(100 * len(ppi_genes) / len(all_pathway_genes), 1),
    "n_hub_genes": len(hub_genes),
    "n_bridge_genes_in_hubs": int(hub_genes['is_bridge'].sum()),
    "top_10_hub_genes": hub_genes['gene'].tolist()[:10],
    "n_communities": len(community_df),
    "n_cross_disease_communities": int(community_df['is_cross_disease'].sum()),
    "network_stats": {
        "n_connected_components": len(components),
        "largest_component_size": len(components[0]),
        "density": round(nx.density(lcc), 4),
        "transitivity": round(nx.transitivity(lcc), 4),
        "avg_clustering": round(nx.average_clustering(lcc), 4),
    },
    "convergence_subnetwork": {
        "n_bridge_genes": convergence_subgraph.number_of_nodes(),
        "n_edges": convergence_subgraph.number_of_edges(),
    },
    "drug_repurposing": {
        "n_interactions": len(drug_interactions) if len(drug_interactions) > 0 else 0,
        "n_unique_drugs": int(drug_interactions['drug'].nunique()) if len(drug_interactions) > 0 else 0,
        "n_genes_with_drugs": int(drug_interactions['gene'].nunique()) if len(drug_interactions) > 0 else 0,
        "top_10_drugs": drug_table_dedup['drug'].tolist()[:10] if len(drug_table_dedup) > 0 else [],
    },
}

# Save summary JSON
with open(OUTPUT_DIR / 'kg_results_summary.json', 'w') as f:
    json.dump(kg_summary, f, indent=2)

# Copy outputs to research-results
import shutil
DEST = Path('../../research-results/knowledge_graph')
DEST.mkdir(parents=True, exist_ok=True)

n_copied = 0
for src_file in OUTPUT_DIR.iterdir():
    if src_file.is_file():
        shutil.copy2(src_file, DEST / src_file.name)
        n_copied += 1

print("=" * 80)
print("KNOWLEDGE GRAPH ANALYSIS — RESULTS SUMMARY")
print("=" * 80)
print(f"\nGene Universe: {kg_summary['n_genes_gmt']} genes from {kg_summary['n_pathways_unique']} pathways")
print(f"PPI Network: {kg_summary['n_ppi_edges']} edges ({kg_summary['ppi_coverage_pct']}% gene coverage)")
print(f"Hub Genes: {kg_summary['n_hub_genes']} (including {kg_summary['n_bridge_genes_in_hubs']} cross-disease bridges)")
print(f"Communities: {kg_summary['n_communities']} ({kg_summary['n_cross_disease_communities']} cross-disease)")
print(f"Convergence Subnetwork: {kg_summary['convergence_subnetwork']['n_bridge_genes']} bridge genes, "
      f"{kg_summary['convergence_subnetwork']['n_edges']} edges")
print(f"Drug Repurposing: {kg_summary['drug_repurposing']['n_unique_drugs']} unique drugs targeting "
      f"{kg_summary['drug_repurposing']['n_genes_with_drugs']} hub genes")

print(f"\nTop 10 Hub Genes: {', '.join(kg_summary['top_10_hub_genes'])}")
if kg_summary['drug_repurposing']['top_10_drugs']:
    print(f"Top 10 Drug Candidates: {', '.join(kg_summary['drug_repurposing']['top_10_drugs'])}")

print(f"\nOutputs: {n_copied} files copied to {DEST}")
print(f"\nOutput files:")
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.is_file():
        size = f.stat().st_size
        print(f"  {f.name} ({size:,} bytes)")

print(f"\n{'=' * 80}")
print("SUCCESS: Knowledge graph analysis complete")
print(f"{'=' * 80}")